In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import PchipInterpolator

# ─────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────
A_H2O     = 2.0e-16
A_CO2     = 7.6e-17
A_CO      = 1.1e-17
CO2_ALPHA = 7.62

# ─────────────────────────────────────────────────────────────
# BAND WINDOWS (µm)
# ─────────────────────────────────────────────────────────────
H2O_WIN = (2.70, 3.50)
CO2_WIN = (4.15, 4.43)
CO_WIN  = (4.63, 4.73)

# ─────────────────────────────────────────────────────────────
# CONTINUUM ANCHORS
# ─────────────────────────────────────────────────────────────
ANCHOR_WINDOWS = [
    (2.42, 2.90),
    (3.55, 3.65),
    (3.85, 4.00),
    (4.10, 4.14),
    (4.46, 4.60),
    (4.82, 5.00),
]

# ─────────────────────────────────────────────────────────────
# SAFE SAVGOL
# ─────────────────────────────────────────────────────────────
def safe_savgol(arr, window, poly):
    n = len(arr)
    if n < poly + 3:
        return arr
    w = min(window, n - 1 if n % 2 == 0 else n)
    w = max(w, poly + 2)
    if w % 2 == 0:
        w -= 1
    return savgol_filter(arr, w, poly)

# ─────────────────────────────────────────────────────────────
# CONTINUUM FIT (LOG-LOG + PCHIP)
# ─────────────────────────────────────────────────────────────
def fit_continuum_loglog(wavelength, flux_smooth):
    anc_wav, anc_flux = [], []

    for lo, hi in ANCHOR_WINDOWS:
        mask = (wavelength >= lo) & (wavelength <= hi)
        if mask.sum() < 3:
            continue

        f_val = np.percentile(flux_smooth[mask], 80)
        w_val = np.median(wavelength[mask])

        if f_val > 0:
            anc_wav.append(w_val)
            anc_flux.append(f_val)

    # fallback if not enough anchors
    if len(anc_wav) < 3:
        log_w = np.log10(wavelength)
        log_f = np.log10(np.clip(flux_smooth, 1e-10, None))
        p = np.polyfit(log_w, log_f, 1)
        return 10 ** np.polyval(p, log_w)

    anc_wav  = np.array(anc_wav)
    anc_flux = np.array(anc_flux)

    srt = np.argsort(anc_wav)
    anc_wav  = anc_wav[srt]
    anc_flux = anc_flux[srt]

    log_anc_w = np.log10(anc_wav)
    log_anc_f = np.log10(anc_flux)

    interp = PchipInterpolator(log_anc_w, log_anc_f, extrapolate=True)
    continuum = 10 ** interp(np.log10(wavelength))

    return continuum

# ─────────────────────────────────────────────────────────────
# MAIN LOOP
# ─────────────────────────────────────────────────────────────
folder_path = r"C:\Users\User\Downloads\New"
output_folder = r"C:\Users\User\Downloads\Spectras"
os.makedirs(output_folder, exist_ok=True)
results = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        try:
            print(f"\nProcessing {file}...")

            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            # 🔥 FORCE NUMERIC
            df['wavelength / micron'] = pd.to_numeric(df['wavelength / micron'], errors='coerce')
            df['Flux / mJy'] = pd.to_numeric(df['Flux / mJy'], errors='coerce')
            df = df.dropna(subset=['wavelength / micron', 'Flux / mJy'])

            wavelength = df['wavelength / micron'].to_numpy(dtype=float)
            flux = df['Flux / mJy'].to_numpy(dtype=float)

            if len(wavelength) < 20:
                print("Too few points — skipped")
                continue

            # sort
            order = np.argsort(wavelength)
            wavelength = wavelength[order]
            flux = flux[order]

            # smooth
            flux_med = (pd.Series(flux)
                        .rolling(7, center=True, min_periods=1)
                        .median()
                        .bfill().ffill().values)

            flux_smooth = safe_savgol(flux_med, 15, 3)
            flux_smooth = np.clip(flux_smooth, 1e-10, None)

            # ─────────────────────────────
            # CONTINUUM
            # ─────────────────────────────
            continuum = fit_continuum_loglog(wavelength, flux_smooth)

            # 🔥 FORCE ABOVE SPECTRUM
            continuum = np.maximum(continuum, flux_smooth)

            # smooth again
            continuum = safe_savgol(continuum, 31, 3)

            continuum = continuum * 1.01
            # ─────────────────────────────
            # PLOT
            # ─────────────────────────────
            plt.figure(figsize=(10,4))

            plt.plot(wavelength, flux, color='gray', alpha=0.3, label='Raw')
            plt.plot(wavelength, flux_smooth, 'k', lw=1.5, label='Smoothed')
            plt.plot(wavelength, continuum, 'r--', lw=1.5, label='Continuum')

            plt.axvspan(*H2O_WIN, color='blue', alpha=0.15, label='H₂O')
            plt.axvspan(*CO2_WIN, color='purple', alpha=0.15, label='CO₂')
            plt.axvspan(*CO_WIN,  color='green', alpha=0.15, label='CO')

            plt.xlim(2.5, 5.0)
            plt.xlabel("Wavelength (µm)")
            plt.ylabel("Flux (mJy)")
            plt.title(file)
            plt.legend()
            plt.grid(alpha=0.2)
            plt.tight_layout()
            png_name = file.replace(".csv", ".png")
            save_path = os.path.join(output_folder, png_name)
            plt.savefig(save_path, dpi=300)
            plt.close() 

        except Exception as e:
            print(f"Error processing {file}: {e}")



Processing 00000_V01_M23_filtered.csv...

Processing 00003_V01_M23_filtered.csv...

Processing 00004_V01_M23_filtered.csv...

Processing 00005_V01_M23_filtered.csv...

Processing 00007_V01_M23_filtered.csv...

Processing 00008_V01_M23_filtered.csv...

Processing 00009_V01_M23_filtered.csv...

Processing 00010_V01_M23_filtered.csv...

Processing 00011_V01_M23_filtered.csv...

Processing 00012_V01_M23_filtered.csv...

Processing 00013_V01_M23_filtered.csv...

Processing 00014_V01_M23_filtered.csv...

Processing 00016_V01_M23_filtered.csv...

Processing 00017_V01_M23_filtered.csv...

Processing 00018_V01_M23_filtered.csv...

Processing 00019_V01_M23_filtered.csv...

Processing 00020_V01_M23_filtered.csv...

Processing 00021_V01_M23_filtered.csv...

Processing 00022_V01_M23_filtered.csv...

Processing 00023_V01_M23_filtered.csv...

Processing 00024_V01_M23_filtered.csv...

Processing 00025_V01_M23_filtered.csv...

Processing 00026_V01_M23_filtered.csv...

Processing 00027_V01_M23_filtered

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import PchipInterpolator

# -----------------------------
# PATHS
# -----------------------------
folder_path = r"C:\Users\User\Downloads\New"

# 🔥 output folder for PNGs
output_dir = os.path.join(folder_path, "plots")
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# CONSTANTS
# -----------------------------
A_H2O = 2.0e-16
A_CO2 = 7.6e-17
A_CO  = 1.1e-17

# -----------------------------
# CONTINUUM WINDOWS
# -----------------------------
ANCHOR_WINDOWS = [
    (2.42, 2.90),
    (3.55, 3.65),
    (3.85, 4.00),
    (4.10, 4.14),
    (4.46, 4.60),
    (4.82, 5.00),
]

# -----------------------------
# SAFE SAVGOL
# -----------------------------
def safe_savgol(arr, window, poly):
    n = len(arr)
    if n < poly + 3:
        return arr
    w = min(window, n - 1 if n % 2 == 0 else n)
    w = max(w, poly + 2)
    if w % 2 == 0:
        w -= 1
    return savgol_filter(arr, w, poly)

# -----------------------------
# CONTINUUM FIT
# -----------------------------
def fit_continuum_loglog(wavelength, flux_smooth):
    anc_wav, anc_flux = [], []

    for lo, hi in ANCHOR_WINDOWS:
        mask = (wavelength >= lo) & (wavelength <= hi)
        if mask.sum() < 3:
            continue

        f_val = np.percentile(flux_smooth[mask], 80)
        w_val = np.median(wavelength[mask])

        if f_val > 0:
            anc_wav.append(w_val)
            anc_flux.append(f_val)

    if len(anc_wav) < 3:
        log_w = np.log10(wavelength)
        log_f = np.log10(np.clip(flux_smooth, 1e-10, None))
        p = np.polyfit(log_w, log_f, 1)
        return 10 ** np.polyval(p, log_w)

    anc_wav = np.array(anc_wav)
    anc_flux = np.array(anc_flux)

    srt = np.argsort(anc_wav)
    anc_wav = anc_wav[srt]
    anc_flux = anc_flux[srt]

    interp = PchipInterpolator(np.log10(anc_wav), np.log10(anc_flux))
    continuum = 10 ** interp(np.log10(wavelength))

    return continuum

# -----------------------------
# MAIN LOOP
# -----------------------------
results = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        try:
            print(f"\nProcessing {file}...")

            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)

            # CLEAN DATA
            df['wavelength / micron'] = pd.to_numeric(df['wavelength / micron'], errors='coerce')
            df['Flux / mJy'] = pd.to_numeric(df['Flux / mJy'], errors='coerce')
            df = df.dropna()

            wavelength = df['wavelength / micron'].to_numpy(float)
            flux = df['Flux / mJy'].to_numpy(float)

            if len(wavelength) < 20:
                continue

            # SORT
            order = np.argsort(wavelength)
            wavelength = wavelength[order]
            flux = flux[order]

            # SMOOTH
            flux_med = pd.Series(flux).rolling(7, center=True).median().bfill().ffill()
            flux_smooth = safe_savgol(flux_med.values, 15, 3)
            flux_smooth = np.clip(flux_smooth, 1e-10, None)

            # CONTINUUM
            continuum = fit_continuum_loglog(wavelength, flux_smooth)

            # 🔥 FORCE ABOVE
            continuum = np.maximum(continuum, flux_smooth)
            continuum = safe_savgol(continuum, 31, 3)
            continuum *= 1.01

            # OPTICAL DEPTH
            ratio = np.clip(flux_smooth / continuum, 1e-6, 1)
            tau = -np.log(ratio)

            # WAVENUMBER
            nu = 10000 / wavelength

            # MASKS
            mask_h2o = (wavelength >= 2.7) & (wavelength <= 3.35)
            mask_co2 = (wavelength >= 4.28) & (wavelength <= 4.30)
            mask_co  = (wavelength >= 4.65) & (wavelength <= 4.70)

            def integrate(tau, nu):
                if len(nu) < 3:
                    return np.nan
                o = np.argsort(nu)
                return np.abs(np.trapz(tau[o], nu[o]))

            area_h2o = integrate(tau[mask_h2o], nu[mask_h2o])
            area_co2 = 7.62 * integrate(tau[mask_co2], nu[mask_co2])
            area_co  = integrate(tau[mask_co], nu[mask_co])

            N_H2O = area_h2o / A_H2O
            N_CO2 = area_co2 / A_CO2
            N_CO  = area_co  / A_CO

            results.append({
                "file": file,
                "N_H2O": N_H2O,
                "N_CO2": N_CO2,
                "N_CO": N_CO
            })

            print(f"H2O={N_H2O:.2e}, CO2={N_CO2:.2e}, CO={N_CO:.2e}")

            # -----------------------------
            # SAVE PLOT
            # -----------------------------
            plt.figure(figsize=(8,4))

            plt.plot(wavelength, flux_smooth, 'k', lw=1, label="Flux")
            plt.plot(wavelength, continuum, 'r--', lw=1.5, label="Continuum")

            plt.axvspan(2.7, 3.35, alpha=0.2, label='H2O')
            plt.axvspan(4.28, 4.30, alpha=0.2, label='CO2')
            plt.axvspan(4.65, 4.70, alpha=0.2, label='CO')

            plt.xlim(2.5, 5.0)

            ymin = min(flux_smooth.min(), continuum.min())
            ymax = max(flux_smooth.max(), continuum.max())
            plt.ylim(ymin*0.95, ymax*1.05)

            plt.xlabel("Wavelength (µm)")
            plt.ylabel("Flux (mJy)")
            plt.title(file)

            plt.legend()
            plt.grid(alpha=0.3)
            plt.tight_layout()

            save_path = os.path.join(output_dir, file.replace(".csv", ".png"))
            plt.savefig(save_path, dpi=300)
            plt.close()

        except Exception as e:
            print(f"Error in {file}: {e}")

# -----------------------------
# FINAL RESULTS
# -----------------------------
df_final = pd.DataFrame(results)

print("\nTOTAL:", len(df_final))
print(df_final.max())


Processing 00000_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00003_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00004_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00005_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00007_V01_M23_filtered.csv...
H2O=nan, CO2=6.50e+17, CO=nan

Processing 00008_V01_M23_filtered.csv...
H2O=1.28e+19, CO2=2.24e+18, CO=5.20e+18

Processing 00009_V01_M23_filtered.csv...
H2O=7.99e+17, CO2=nan, CO=nan

Processing 00010_V01_M23_filtered.csv...
H2O=1.85e+18, CO2=nan, CO=nan

Processing 00011_V01_M23_filtered.csv...
H2O=3.80e+18, CO2=nan, CO=nan

Processing 00012_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00013_V01_M23_filtered.csv...
H2O=2.89e+18, CO2=8.77e+17, CO=4.57e+18

Processing 00014_V01_M23_filtered.csv...
H2O=2.22e+18, CO2=nan, CO=nan

Processing 00016_V01_M23_filtered.csv...
H2O=6.78e+18, CO2=1.10e+18, CO=5.00e+18

Processing 00017_V01_M23_filtered.csv...
H2O=6.01e+18, CO2

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from scipy.interpolate import PchipInterpolator

# -----------------------------
# PATHS
# -----------------------------
folder_path = r"C:\Users\User\Downloads\CSV Files"

output_dir = os.path.join(folder_path, "plots2")
os.makedirs(output_dir, exist_ok=True)

# -----------------------------
# CONSTANTS
# -----------------------------
A_H2O = 2.0e-16
A_CO2 = 7.6e-17
A_CO  = 1.1e-17

# -----------------------------
# CONTINUUM WINDOWS
# -----------------------------
ANCHOR_WINDOWS = [
    (2.42, 2.90),
    (3.55, 3.65),
    (3.85, 4.00),
    (4.10, 4.14),
    (4.46, 4.60),
    (4.82, 5.00),
]

# -----------------------------
# SAFE SAVGOL
# -----------------------------
def safe_savgol(arr, window, poly):
    n = len(arr)
    if n < poly + 3:
        return arr
    w = min(window, n - 1 if n % 2 == 0 else n)
    w = max(w, poly + 2)
    if w % 2 == 0:
        w -= 1
    return savgol_filter(arr, w, poly)

# -----------------------------
# CONTINUUM FIT
# -----------------------------
def fit_continuum_loglog(wavelength, flux_smooth):
    anc_wav, anc_flux = [], []

    for lo, hi in ANCHOR_WINDOWS:
        mask = (wavelength >= lo) & (wavelength <= hi)
        if mask.sum() < 3:
            continue

        f_val = np.percentile(flux_smooth[mask], 80)
        w_val = np.median(wavelength[mask])

        if f_val > 0:
            anc_wav.append(w_val)
            anc_flux.append(f_val)

    if len(anc_wav) < 3:
        log_w = np.log10(wavelength)
        log_f = np.log10(np.clip(flux_smooth, 1e-10, None))
        p = np.polyfit(log_w, log_f, 1)
        return 10 ** np.polyval(p, log_w)

    anc_wav = np.array(anc_wav)
    anc_flux = np.array(anc_flux)

    srt = np.argsort(anc_wav)
    anc_wav = anc_wav[srt]
    anc_flux = anc_flux[srt]

    interp = PchipInterpolator(np.log10(anc_wav), np.log10(anc_flux))
    continuum = 10 ** interp(np.log10(wavelength))

    return continuum

# -----------------------------
# MAIN LOOP
# -----------------------------
results = []

for file in os.listdir(folder_path):
    if file.endswith(".csv"):
        try:
            print(f"\nProcessing {file}...")
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)
            df['wavelength / micron'] = pd.to_numeric(df['wavelength / micron'], errors='coerce')
            df['Flux / mJy'] = pd.to_numeric(df['Flux / mJy'], errors='coerce')
            df = df.dropna()

            wavelength = df['wavelength / micron'].to_numpy(float)
            flux = df['Flux / mJy'].to_numpy(float)

            if len(wavelength) < 20:
                continue

            # SORT
            order = np.argsort(wavelength)
            wavelength = wavelength[order]
            flux = flux[order]

            # SMOOTH
            flux_med = pd.Series(flux).rolling(7, center=True).median().bfill().ffill()
            flux_smooth = safe_savgol(flux_med.values, 15, 3)
            flux_smooth = np.clip(flux_smooth, 1e-10, None)

            # CONTINUUM
            continuum = fit_continuum_loglog(wavelength, flux_smooth)

            # 🔥 FORCE ABOVE
            continuum = np.maximum(continuum, flux_smooth)
            continuum = safe_savgol(continuum, 31, 3)
            continuum *= 1.01

            # OPTICAL DEPTH
            ratio = np.clip(flux_smooth / continuum, 1e-6, 1)
            tau = -np.log(ratio)

            # WAVENUMBER
            nu = 10000 / wavelength

            # MASKS
            mask_h2o = (wavelength >= 2.7) & (wavelength <= 3.35)
            mask_co2 = (wavelength >= 4.28) & (wavelength <= 4.30)
            mask_co  = (wavelength >= 4.65) & (wavelength <= 4.70)

            def integrate(tau, nu):
                if len(nu) < 3:
                    return np.nan
                o = np.argsort(nu)
                return np.abs(np.trapz(tau[o], nu[o]))

            area_h2o = integrate(tau[mask_h2o], nu[mask_h2o])
            area_co2 = 7.62 * integrate(tau[mask_co2], nu[mask_co2])
            area_co  = integrate(tau[mask_co], nu[mask_co])

            N_H2O = area_h2o / A_H2O
            N_CO2 = area_co2 / A_CO2
            N_CO  = area_co  / A_CO
            results.append({
                "file": file,
                "N_H2O": N_H2O,
                "N_CO2": N_CO2,
                "N_CO": N_CO})

            print(f"H2O={N_H2O:.2e}, CO2={N_CO2:.2e}, CO={N_CO:.2e}")
            plt.figure(figsize=(8,4))
            plt.plot(wavelength, flux_smooth, 'k', lw=1, label="Flux")
            plt.plot(wavelength, continuum, 'r--', lw=1.5, label="Continuum")
            plt.axvspan(2.7, 3.35, alpha=0.2,color='blue', label='H2O')
            plt.axvspan(4.28, 4.30, alpha=0.2, color='purple',label='CO2')
            plt.axvspan(4.65, 4.70, alpha=0.2, color='green',label='CO')
            plt.xlim(2.5, 5.0)
            ymin = min(flux_smooth.min(), continuum.min())
            ymax = max(flux_smooth.max(), continuum.max())
            plt.ylim(ymin*0.95, ymax*1.05)
            plt.xlabel("Wavelength (µm)")
            plt.ylabel("Flux (mJy)")
            plt.title(file)
            plt.legend()
            plt.grid(alpha=0.3)
            def format_cd(val):
                if np.isnan(val):
                    return "0"
                else:
                    return f"{val/1e18:.2f} ×10¹⁸"
            textstr = (
                f"H2O = {format_cd(N_H2O)} cm⁻²\n"
                f"CO2 = {format_cd(N_CO2)} cm⁻²\n"
            f"CO  = {format_cd(N_CO)} cm⁻²"
            )
            plt.text(
                0.02, 0.98, textstr,
                transform=plt.gca().transAxes,
                fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round', facecolor='white', alpha=0.7)
            )
            plt.tight_layout()
            save_path = os.path.join(output_dir, file.replace(".csv", ".png"))
            plt.savefig(save_path, dpi=300)
            plt.close()


        except Exception as e:
            print(f"Error in {file}: {e}")

# -----------------------------
# FINAL RESULTS
# -----------------------------
df_final = pd.DataFrame(results)

print("\nTOTAL:", len(df_final))
print(df_final.max())


Processing 00000_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00003_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00004_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00005_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00007_V01_M23_filtered.csv...
H2O=nan, CO2=6.50e+17, CO=nan

Processing 00008_V01_M23_filtered.csv...
H2O=1.28e+19, CO2=2.24e+18, CO=5.20e+18

Processing 00009_V01_M23_filtered.csv...
H2O=7.99e+17, CO2=nan, CO=nan

Processing 00010_V01_M23_filtered.csv...
H2O=1.85e+18, CO2=nan, CO=nan

Processing 00011_V01_M23_filtered.csv...
H2O=3.80e+18, CO2=nan, CO=nan

Processing 00012_V01_M23_filtered.csv...
H2O=nan, CO2=nan, CO=nan

Processing 00013_V01_M23_filtered.csv...
H2O=2.89e+18, CO2=8.77e+17, CO=4.57e+18

Processing 00014_V01_M23_filtered.csv...
H2O=2.22e+18, CO2=nan, CO=nan

Processing 00016_V01_M23_filtered.csv...
H2O=6.78e+18, CO2=1.10e+18, CO=5.00e+18

Processing 00017_V01_M23_filtered.csv...
H2O=6.01e+18, CO2